# report13 — 자유공간 드론 검지거리 (FS-1)

**핵심.** 앞 리포트의 표적(SBR σ)·신호(Sionna PHY)·검출기(레이더 DSP)를 **자유공간(FS-1)** 무대에 올려, **Pd 0.9 @ 셀당 Pfa 1e-4·단일 CPI·이상 DPI소거 기준 검지거리**를 닫힌형+측정으로 낸다. 헤드라인은 LTE CRS(L1)·전력규약 equal_psd·T_CPI — ms 에서 대표 기체(mavic4pro) R_eq **4290 m**(중점수평 d 4282 m). ★새 정보는 σ(음의 앙각)와 **기하(Cassini·비단조·천장)**다.

| 이 리포트의 척추 |  |
|---|---|
| **① Sionna 의 공백** | Sionna 는 경로·도플러·렌더(RT)와 파형·채널(PHY)은 담보하나, **표적 σ 는 산란적분 부재로 못 내고**(report06), **패시브 레이더 검출 파이프라인은 아예 없다**. FS-1 에는 챔버의 정적 클러터가 **애초에 없어**(죽은 파라미터), 그 자리를 §6 '벽 지도'(열잡음·ADC·DPI잔류·range-walk)가 대신한다. |
| **② 선행 연구의 방식** | 패시브 바이스태틱 드론탐지 선행은 표적 산란을 외부에서 구해 채널에 주입(h=h_bg+h_target)하고 표준 검출 체인(ECA/CAF/CFAR)으로 잡는다. 조사한 선행은 대개 **단일 조명원·챔버 없는 실외**라, **5기종×9모드를 한 규약으로 검지거리 비교한 사례는 확인하지 못했다**(부재증명 아님). |
| **③ 쓴 라이브러리·결합** | σ 는 **SBR+PO**(Mitsuba 광선+PO 표면적분), 에코 지연펄스정형은 **Sionna PHY** 커널, 검출은 **pyAPRiL 로 검증된 ECA/CFAR 체인**(radar-dsp), 렌더는 **Sionna RT**(sionna-render). 새 파형·산란 엔진을 만들지 않고 검증된 조각만 자유공간 예산으로 결합한다. |
| **④ 검증** | 9모드 SNR90 동일성(FS-0)·닫힌형 링크버짓 대조(k_mode 잔차 +0.000 dB)·ECA바닥(ridge=0 잔류≈0)·이등분선↔멀티스태틱 Δσ(β)·나이퀴스트폴드·챔버 되돌리기. 절대 σ 는 실측 문헌 드론 RCS 로 앵커(report08). |

---


## 📋 이 결과가 어디서 어떻게 나왔나

> 이 절은 **직접 참여하지 않은 사람도 출처를 따라가고 재현할 수 있도록** 넣었습니다. 버전·GPU 는 노트북 생성 시점에 **실제로 읽어온 값**입니다.

### 1️⃣ 무엇을 참고했나

| 항목 | 출처 | 성격 |
|---|---|---|
| 검지거리·커버리지·감도·벽지도 | outputs/report13_freespace.json | 닫힌형 전파 + 몬테카를로 문턱측정 |
| 표적 밝기 σ (음의 앙각 격자·멀티스태틱) | outputs/report13_sigma_grid.json → SBR(Mitsuba+PO) | SBR+PO 배치계산 (GPU) |
| 검증(닫힌형·ECA바닥·FS-0·챔버) | outputs/verify_freespace.json | 교차검증 |
| 절대 σ 앵커 | 공개 문헌 드론 RCS (report08) | 외부 앵커 (±수 dB) |

### 2️⃣ 어떤 도구가 무엇을 했나 — **Sionna 내부인가, 우리가 짠 건가**

| 도구 | 하는 일 | 어디서 도는가 |
|---|---|---|
| `sbr` | SBR (`src/rcs_sbr.py`) — **Mitsuba 광선 + PO 표면적분**으로 RCS. 가림(occlusion) 포함 | 🟡 **우리가 짰다** — 다만 광선추적은 Sionna 가 쓰는 **Mitsuba 3 엔진 그대로** (GPU). Sionna 에 RCS 솔버가 없기 때문 |
| `sionna-phy` | Sionna PHY (`ofdm`/`nr`/`channel`) — OFDM 변복조 · 3GPP 뉴머롤로지 · RT 경로를 신호에 적용 | 🟢 **Sionna 내부** (PyTorch 백엔드, GPU) |
| `sionna-render` | Sionna RT `Scene.render_to_file()` — 씬·**추적된 광선**·라디오맵을 사진처럼 렌더 | 🟢 **Sionna 내부** (Mitsuba 3 경로추적 렌더러, GPU) |
| `radar-dsp` | 레이더 신호처리 (`src/passive_process.py`) — ECA(직접파 제거) · 거리-도플러 · CA-CFAR | 🔴 **별도** (numpy, CPU). **Sionna 에 레이더 DSP 는 없다** |
| `matplotlib` | matplotlib — 도표·그래프 | 🔴 **별도** (CPU). 계산 결과를 *그리기만* 한다 |

> 🔑 **이 구분이 이 프로젝트에서 가장 자주 오해받는 지점입니다.**
> - **전파**(경로·지연·도플러·렌더·라디오맵)는 🟢 **Sionna 가** 합니다.
> - **표적 RCS** 는 🟡 우리가 얹은 **PO(물리광학 표면적분)** 가 냅니다 — Sionna 기본 solver 엔 이 산란적분이 없어 경로 이득만 줄 뿐 RCS 를 못 내기 때문입니다. 광선을 쏴 조명면·가림을 찾는 **SBR** 은 Sionna 의 **Mitsuba 3 엔진을 그대로** 쓰고, 그 위에 **PO 적분만 우리가** 얹습니다(SBR+PO).
> - **레이더 신호처리**(ECA/CFAR)는 🔴 우리가 짰습니다 — Sionna 에 레이더 DSP 가 없습니다.

### 3️⃣ 라이브러리 (실행 시점 **실측** 버전)

| 라이브러리 | 버전 | 무엇에 쓰나 |
|---|---|---|
| `sionna` | 2.0.1 | 광선추적(RT) + PHY(OFDM/NR/채널) — **이 프로젝트의 중심** |
| `mitsuba` | 3.8.0 | Sionna RT 의 렌더러·광선추적 백엔드 (OptiX, GPU). SBR 도 이걸 쓴다 |
| `torch` | 2.12.1 | Sionna PHY 백엔드 — ⚠ Sionna 2.0 은 TensorFlow 가 아니라 **PyTorch** |
| `numpy` | 2.5.0 | 수치 계산 전반 |
| `matplotlib` | 3.11.0 | 도표 |
| `Pillow` | 12.2.0 | GIF 합성 |

### 4️⃣ 어디서 돌렸나

- **Python** 3.12.13 · Linux 5.15.0-136-generic
- **GPU** — `src/gpu.py` 가 **여유 메모리를 보고 자동 선택**합니다 (하드코딩 없음):
  - 0, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 1, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 2, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 3, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
- `CUDA_VISIBLE_DEVICES` = (고정 안 함 — src/gpu.py 가 여유 메모리 보고 자동 선택)

- **계산 비용**: σ격자 GPU 3~4장 ~1.5h · 문턱MC GPU 1~2장 ~2~4h · 렌더 1장 ~60min · 그림/노트북 CPU ~30min

### 5️⃣ 어떻게 다시 돌리나 (재현)

```bash
# σ격자(GPU 3~4장 ~1.5h)
SIONNA2_GPU=3 PYTHONPATH=src:benchmark python src/experiment_freespace_sigma.py
# 문턱·거리역해·감도·벽지도(GPU 1~2장 ~2~4h)
PYTHONPATH=src:benchmark python src/experiment_freespace_range.py --stage=all
# 검증 / 그림 / RT렌더 / 이 노트북
PYTHONPATH=src:benchmark python benchmark/verify_freespace.py
PYTHONPATH=src:benchmark python src/viz_report13.py
SIONNA2_GPU=3 PYTHONPATH=src:benchmark python src/render_report13.py --which all
PYTHONPATH=src:benchmark python src/make_notebook13.py
```

### 6️⃣ 본문 숫자는 어디서 오나

이 노트북의 **숫자는 손으로 적지 않았습니다.** 측정 스크립트가 JSON 을 남기고, 노트북 생성기(`src/make_notebook*.py`)가 그 JSON 을 읽어 본문에 주입합니다. → **그림과 글이 어긋날 수 없습니다.** 숫자가 이상하면 JSON 을 보세요.

### 7️⃣ 무엇이 산출되나

| 산출물 | 무엇 |
|---|---|
| `outputs/report13_freespace.json` | 검지거리·커버리지·감도·벽지도(모든 본문 숫자의 출처) |
| `outputs/report13_sigma_grid.json` | 음의 앙각 σ 격자·멀티스태틱·D/λ·공통반송파 반사실 |
| `outputs/verify_freespace.json` | 닫힌형·ECA바닥·FS-0·챔버 되돌리기 검증 |
| `outputs/figures/report13_*.png` | 그림 F1~F16 |
| `outputs/renders/anim/r13_*.gif` | RT/도식 GIF R1~R8 |
| `outputs/renders/r13_*.png` | RT 스틸 22장 |

### 8️⃣ ⚠️ 믿으면 안 되는 것 (신뢰 경계)

> 정직함이 이 프로젝트의 규칙입니다. **아래는 이 리포트가 보장하지 않는 것들입니다.**

- **"실제 검지거리 N m"라고 말할 수 없다.** EIRP·G_rx(10dBi)·NF(5dB)·마스트25m·RX3m·고도·속도가 **전부 선언값**(근거문서 없음, 코드 주석이 유일). 결과는 '선언한 예산 아래의 거리'다.
- **이 편만 FS-1(자유공간)이다 — 챔버 상한선이 아니라 기준정규화다.** 평면지면만으로 F⁴∈[−20,+11.5]dB(거리 0.32~1.94배) 변조(FS-3). 실외는 클러터·다중경로·간섭까지 더해 **다를** 뿐 반드시 나쁘진 않다.
- **낙관 방향**: (a) 무클러터·무지면(FS-1)은 상한 아님(FS-3가 방향 보임) (b) 기준채널=무잡음 full-waveform(현실 2채널 CAF 아님) (c) 비요동은 **CPI 내부**만(inter-look 은 E_ψ[Pd]로 포함) (d) SBR+PO few-λ 낙관.
- **σ 절대값 ±수 dB 앵커.** 격자불확실성·편파 없음(스칼라|Γ|)·PTD/크리핑파 없음. Mavic4Pro·Matrice4E 실측RCS 문헌 없음. 신뢰도가 (기종×밴드)마다 다름(mini5pro@LTE 2.32λ 최악) — D/λ 지도로.
- **널깊이·σ_min·aspect-peak 인용 금지.**
- **바이스태틱 σ는 β≲90°만 유효**(전방산란 σ≡0). 근거리·큰 L 은 걸림 — 해칭·수치금지. 상반성 깊은널 rms 5~9dB.
- **ECA는 무한이상화가 아니다.** 정본 ridge_rel=0 은 측정상 사실상 이상적(잔류≈0). 진짜 낙관은 **실장 유한 소거깊이**(통상 40~90dB)·위상잡음·상호변조가 모델에 없다는 것 → `dpi_residual` 감도축으로 정량화.
- **ADC는 1차근사.** 균일양자화+백색. AGC·SFDR·상호변조 미모델. '12-bit 가 벽'이라 단정 안 함(조건부).
- **검출기 형상 둘·성격 다름.** S-G(게이트)=탐색없는 거리, S-W(전체창)=CPI당 오경보 수(분해능셀 기준). 검출정의=표적근방 문턱초과(전역 argmax 아님).
- **N^¼ 이득은 이상적 상한**(완전코히런트·완벽조향·무상관). 개구는 밴드마다 다름.
- **기준=full-waveform capture 정본** — 패시브인데 상시신호만은 아니라는 유보 승계(pilot-only 병기).
- **CAF 2채널 승격 안 함** — 이상적 레퍼런스 정합필터 SNR(report12 동일 열린과제).
- **문헌 실적치와 우열비교 안 함**(`prior_compare` 조건열 강제).
- **호버·저속·앨리어싱 미검출은 거리무관.** 유휴 5G SSB(PRF50Hz)는 v=5m/s 전 헤딩 접힘 — 거리 이전에 도플러로 죽는다(하한 hover-blind + 상한 나이퀴스트 둘 다).
- **자세=수평(roll=pitch=0)·yaw만.** 실전 진비행 pitch 10~30° 미모델 — **모른다**.
- **el 격자 보간**: el≤−15° 는 β>90° 겹쳐 표본 적음. 헤드라인 el≈−2° 라 부호정정 효과는 헤드라인서 작다(d≲400m 한정). el>0(공중조명원) 범위 밖.
- **원거리장**: s1000plus@5.21G 63.1m 최악. 그 아래 σ 인용 안 함.
- **'5기종×9모드 한 규약 선행사례 확인 못함'까지만** — 부재증명 아님, '최초' 안 씀.
- **단일 CPI·단일 표적.** 스캔누적·M-of-N·트래킹 없음(future work). 상호그림자 없음.
- **대기감쇠·전파수평선은 숫자와 함께 무시**(FS-1엔 항 자체 없음, FS-3서만): 산소흡수 5km<0.1dB, 수평선≈39km≫20km.
- **비단조·천장.** SNR(d)는 φ의 다수서 내부최대 — 이분법 무효, 최외곽교차로 해결. 천장보다 어두운 헤딩은 어떤 거리서도 미검출(거리문제 아니라 해 없음).
- **비선형.** R∝σ^¼ 는 R_eq 축서만 정확. d 축 국소지수 1.03~4.00 — dB↔배율 환산 금지. k_mode(곱셈상수)는 이 기하 비선형 흡수 못함.
- **동일채널 간섭** 한 항으로 거리 한자릿수 감소 가능(INR20dB 이웃셀) — 감도축으로만, 정본엔 미포함.
- **파형↔반송파 결속.** 등전력만으론 파형효과 미분리 — 공통3.5GHz 반사실로 5항 분해. '어느 파형이 멀리 보나'는 밴드·σ(f)·반복률·fs선택의 합성이다.

### 9️⃣ 앞뒤 리포트

| 리포트 | 관계 |
|---|---|
| report06~08 | 표적 σ (Sionna 한계·SBR·RCS 결과) — 이 편이 소비하는 밝기 |
| report12 | 챔버 안 9모드 벤치마크 — 이 편은 그 검출기를 **자유공간 예산**으로 옮긴 것 |
| report09~11 | 바닥유령·CFAR교정·관측성 — FS-3 정직성 점검·벽 지도의 뿌리 |

<details><summary><b>🔤 용어집 — 모르는 말이 나오면 여기</b> (클릭)</summary>

| 용어 | 뜻 |
|---|---|
| **FS-0/1/2/3 (가정 사다리)** | FS-0=표적에코+열잡음 상한, **FS-1=+직접파·ECA·0도플러가드(헤드라인)**, FS-2=+ADC양자화, FS-3=+평면지면 2-ray(정직성 점검). FS-1 은 **상한이 아니라 기준정규화**다. |
| **R1 / R2 / R_b / R_eq** | R1=TX→표적, R2=표적→RX, R_b=R1+R2−L(바이스태틱 거리), **R_eq=√(R1·R2)**(σ^¼ 이 정확한 축). d=중점 수평거리(장면을 펼치는 축). |
| **커버리지 C(d) / E_ψ[Pd]** | C(d)=거리 d 에서 Pd≥0.9 인 **헤딩의 비율**. E_ψ[Pd]=헤딩 평균 검출확률. 단일 '검지거리'가 아니라 **분포**로 말한다. |
| **점유·전력규약 (equal_psd/equal_total/deploy)** | 같은 조명을 어떻게 정규화하나. **equal_psd**(정본)=per-RE 송신전력 동일 → 시간희소 SSB(G1)의 낮은 평균방사전력을 물리반영. deploy=배치현실(유휴 gNB≠풀 gNB). |
| **5G 이중고 (SSB)** | 유휴 5G SSB(G1)는 반복률 50Hz·협대역 7.2MHz라 **거리도 속도도 나쁘다** — 상시 3인방 중 꼴찌. PRS(측위세션)면 전대역이지만 상시는 아니다. |
| **dpi_residual (실장 유한 소거)** | 실제 ECA·아날로그 소거는 무한이 아니다(40~90dB). 그 잔류 직접파가 만드는 잡음바닥 → 자유공간 최대 벽. |

</details>

---


## §0 — 왜 챔버를 벗어나나 (가정 사다리 FS-0/1/2/3)

report01~12 는 30×20×11 m 무향실 **안**의 실험이었다. 이 편은 **반사체가 없는 자유공간(FS-1)** 을 가정한다. 다만 자유공간은 **상한선이 아니라 기준정규화**다 — 평면지면 하나만 놓아도(FS-3) 에코가 F⁴∈[−20,+11.5]dB 로 흔들린다(정직성 점검). 사다리:

| 단계 | 포함 | 재는 것 | 헤드라인 |
|---|---|---|---|
| **FS-0** | 표적에코+열잡음 | 열잡음 상한, 9모드 SNR90 동일성 | — |
| **FS-1** | +직접파·ECA(ridge=0)·0도플러가드·이상 DPI소거 | **보고하는 R90** | ✅ |
| **FS-2** | +ADC 양자화 | 동적범위 벽(조건부) | — |
| **FS-3** | +평면지면 2-ray·fresnel·대기감쇠 | 정직성 점검(자유공간은 상한 아님) | — (동반) |


![자유공간 바이스태틱 기하 — 거리 셋, 측정 하나](outputs/figures/report13_geometry.png)

![챔버가 사라진 순간 — 자유공간 바이스태틱 기하 궤도](outputs/renders/anim/r13_geometry_orbit.gif)

<sub>TX 마스트(25m)·RX(3m)·표적. 바닥·벽 없음. 실제 베이스라인은 압축했고 프레임에 'scale break' 를 표기한다.</sub>

![report01~12(챔버 R_b≈— m)가 어디 있고, 평면지면(FS-3)이 무엇을 할지](outputs/figures/report13_chamber_vs_freespace_and_ground.png)

## §1 — 거리가 셋이다 (R1 · R2 · R_b · R_eq)

바이스태틱은 왕복이 아니다. **R1**(TX→표적)·**R2**(표적→RX)·**R_b=R1+R2−L**(바이스태틱 거리)가 다르고, σ→거리 사상이 정확한 축은 **R_eq=√(R1·R2)** 다. 같은 R_b 는 TX·RX 를 초점으로 한 타원, 같은 SNR 은 **Cassini oval** 을 그린다 — 이 기하가 커버리지를 접는다.

> **헤드라인.** EIRP — dBm(in-burst peak)·전력규약 equal_psd·T_CPI — ms·Pd 0.9 @ 셀당 Pfa 1e-4·단일 CPI·이상 DPI소거(∞ dB) 기준, 자유공간(FS-1) 검지거리는 **LTE CRS(L1)**·대표 기체 `mavic4pro` 에서 **R_eq 4290 m**(중점수평 d 4282 m) [자세 커버리지 C25–C50: 4282–4282 m], 헤딩평균 검출확률 E_ψ[Pd]=0.65. **실장 소거 60 dB 기준 1816 m.**

![거리 셋과 하나의 측정 — iso-R_b 타원 + Cassini oval](outputs/figures/report13_geometry.png)

![베이스라인 L 스윕 — Cassini 단일→이엽 (β>90° 해칭)](outputs/renders/anim/r13_cassini_baseline.gif)

<sub>기하가 커버리지를 접는다 — L 을 늘리면 등감도선이 단일 오벌에서 두 잎으로 갈린다.</sub>

## §2 — 표적 (5기종 · σ 는 방위와 앙각의 함수)

5기종은 **target_extent(메쉬 bbox 최대 수평치수) 오름차순**으로 고정한다: `mini5pro < phantom4 < mavic4pro < matrice4e < s1000plus`. σ 는 단일 숫자가 아니라 **(방위, 앙각)의 함수**다. 지상 TX/RX + 공중 표적이라 이등분선 앙각은 **전 구간 음수** — 우리는 드론의 **배(belly)** 를 본다.

![5기종 동일 축척 1열 궤도(target_extent 순)](outputs/renders/anim/r13_five_lineup_orbit.gif)

<sub>메쉬·재질색·실제 크기 대비. 스케일바로 1 m 를 표시한다.</sub>

![RCS σ(방위, 앙각≤0) — 5기종 @3.5 GHz (배를 올려다본다)](outputs/figures/report13_sigma_grid_5.png)

![기종별 자세각 다이얼 — yaw 0→360°(프롭 위상), 시선은 음의 앙각](outputs/renders/anim/r13_aspect_mavic4pro.gif)

<sub>좌: RT 드론 회전(이 모듈). 우 σ(ψ) 극좌표 다이얼은 matplotlib(viz_report13)가 합성한다.</sub>

## §3 — 언제 보이나 (Pd(SNR) 측정 · 도플러 오프셋 · argmax 아님)

검출 문턱은 **측정한다**(가정 12dB 아님): 몬테카를로로 Pd0.9 가 되는 RD 출력 SNR 을 찾고, Pfa 는 셀당 1e-4 로 경험 교정한다. 검출 정의는 **표적 근방(±2) 문턱 초과**(전역 argmax 아님). 전이곡선은 표적의 **도플러 오프셋**에 의존하므로 여러 오프셋 빈에서 잰다.

> **5G 이중고.** 유휴 5G SSB(G1)는 반복률 50 Hz·협대역 7.2 MHz라 거리도 속도도 나쁘다: v=5 m/s 에서 헤딩의 약 **—%** 가 도플러 블라인드, 나머지도 M≈— 로 코히런트 이득이 낮아 상시 3인방 중 꼴찌다.

![측정한 Pd(SNR) + 경험 Pfa + 도플러 오프셋 의존](outputs/figures/report13_detector.png)

![거리-도플러 맵의 표적 봉우리 소멸 — d 200 m → 5 km](outputs/renders/anim/r13_rd_recede.gif)

<sub>실 MC RD맵(`passive_process.range_doppler`): 표적이 멀어질수록 거리빈이 밀려나가고 봉우리가 잡음바닥으로 가라앉는다 — 검지거리가 '거리 문제'로 보이는 물리.</sub>

## §4 — 몇 m 인가 (R90_C50 + E_ψ[Pd] 병기 · 5G 꼴찌)

단일 '검지거리'는 없다. **R90_C50**(헤딩의 50%가 Pd≥0.9 인 최외곽 거리)와 **E_ψ[Pd]**(헤딩 평균 검출확률)를 **함께** 말한다. 커버리지 밴드는 C25–C50 이며, 커버리지 천장 C_max=1−b_blind 상대다(b_blind>10%면 C10 은 미정의).

> 전력규약을 배치현실(deploy)로 바꾸면 **유휴 5G(NR G1)가 무너진다**: equal_psd 대비 평균 방사전력이 약 — dB 낮다(유휴 gNB ≠ 풀 gNB). 상시 3인방 순위 전복은 이 프로젝트의 핵심 서사(5G 이중고)다.

![자유공간 검지거리 — 5기종 × 상시 3인방 (채운 띠=C25–C50, 캡=CI95)](outputs/figures/report13_range_bars.png)

![R90 행렬 — 5기종 × 9모드 (equal-PSD | deploy-EIRP)](outputs/figures/report13_matrix.png)

![커버리지 C(d) 와 평균 Pd — 헤딩 비율 vs 헤딩평균](outputs/figures/report13_coverage_curves.png)

## §5 — 왜 밴드가 넓나 (σ 파이프라인 추정량 · R_eq 축 · 분산이 결론)

검지거리 밴드의 정체는 **σ의 자세 분산**이다. 밴드는 파이프라인과 **동일한 추정량**(5점 m² 평균, 음의 앙각)으로 재산출하고, dB↔배율 환산은 하지 않는다(국소지수 n_local 병기, R_eq 축서만 σ^¼ 정확).

> **분산이 결론.** d≳1 km 에서는 **자세**가, 그 아래에서는 **기하(φ)**가 분산을 지배한다. 헤드라인 지점의 국소지수 n_local≈3.99, 앙각 el≈-0.6°. (전 캡션 φ=90° 명시.)

![RCS 분포에서 거리 분포로 (파이프라인 추정량)](outputs/figures/report13_sigma_to_range.png)

![표적 헤딩 대 거리 — 자세·도플러 블라인드·앨리어싱이 한 축에서 작용](outputs/figures/report13_heading_footprint.png)

![배를 올려다본다 — σ vs (음의) 이등분선 앙각, 헤드라인의 위치](outputs/figures/report13_elevation.png)

## §6 — 무엇이 거리를 정하나 (벽 지도: 열잡음·ADC·DPI잔류·range-walk·INR)

**어느 하나를 벽으로 세우지 않는다** — 셀마다 `limit` 라벨이 붙는다(`thermal/adc/dpi_residual/walk/farfield/beta/nonlinear`). 조명 전력은 R∝EIRP^¼, CPI 는 R∝T^¼ 로만 사는데, ADC·DPI잔류 바닥은 EIRP 와 무관하다. **진짜 큰 벽은 dpi_residual**(실장 유한 소거)다 — 소거 60 dB 기준 헤드라인은 1816 m 로 준다.

range-walk 은 대역폭이 넓을수록 이르게 온다(ΔR_b=c/B → T_max=ΔR_b/v). 동일채널 간섭(INR)은 이웃 셀 하나로 거리를 한 자릿수 줄일 수 있어 감도축으로만 냈다.

![거리는 조명 전력의 네제곱근으로 — ADC·DPI잔류 바닥은 그렇지 않다](outputs/figures/report13_eirp_ladder.png)

![어느 벽이 붙나? thermal / ADC / DPI-residual / walk, 밴드·베이스라인별](outputs/figures/report13_walls.png)

![긴 CPI 는 R∝T^¼ 를 사지만 — range-walk(가장 좁은 벽)까지만](outputs/figures/report13_cpi_walk.png)

![대역폭은 위치를 사지 검출을 사지 않는다](outputs/figures/report13_resolution_vs_range.png)

![헤드라인 앙각 효과가 작음을 정직 표기](outputs/figures/report13_elevation.png)

## §7 — 믿어도 되나 (닫힌형 대조 · ECA바닥 · 멀티스태틱 · 챔버 되돌리기)

측정 파이프라인은 닫힌형 링크버짓과 ±0.1 dB 로 맞는다(k_mode 잔차 +0.000 dB). ECA 바닥은 ridge_rel=0 에서 잔류≈0(설계 원안 1e-6 은 오히려 +25 dB 누설). 이등분선↔멀티스태틱 Δσ(β)·나이퀴스트 폴드·FS-0 9모드 동일성·챔버 되돌리기(F16)까지 교차검증한다.

![검증 — 닫힌형 vs 측정, ECA 바닥(ridge=0), 이등분선 vs 멀티스태틱](outputs/figures/report13_verify.png)

![report01~12 의 위치와 평면지면(FS-3)이 할 일 — 정직성](outputs/figures/report13_chamber_vs_freespace_and_ground.png)

## §8 — 말할 수 없는 것 (살아남은 한계 — 전량)

> 정직함이 이 프로젝트의 규칙이다. 아래는 이 리포트가 **보장하지 않는 것들**이다(요약; 상세는 상단 프로venance §8️⃣ 과 동일).

1. **"실제 검지거리 N m"라고 말할 수 없다.** EIRP·G_rx(10dBi)·NF(5dB)·마스트25m·RX3m·고도·속도가 **전부 선언값**(근거문서 없음, 코드 주석이 유일). 결과는 '선언한 예산 아래의 거리'다.
2. **이 편만 FS-1(자유공간)이다 — 챔버 상한선이 아니라 기준정규화다.** 평면지면만으로 F⁴∈[−20,+11.5]dB(거리 0.32~1.94배) 변조(FS-3). 실외는 클러터·다중경로·간섭까지 더해 **다를** 뿐 반드시 나쁘진 않다.
3. **낙관 방향**: (a) 무클러터·무지면(FS-1)은 상한 아님(FS-3가 방향 보임) (b) 기준채널=무잡음 full-waveform(현실 2채널 CAF 아님) (c) 비요동은 **CPI 내부**만(inter-look 은 E_ψ[Pd]로 포함) (d) SBR+PO few-λ 낙관.
4. **σ 절대값 ±수 dB 앵커.** 격자불확실성·편파 없음(스칼라|Γ|)·PTD/크리핑파 없음. Mavic4Pro·Matrice4E 실측RCS 문헌 없음. 신뢰도가 (기종×밴드)마다 다름(mini5pro@LTE 2.32λ 최악) — D/λ 지도로.
5. **널깊이·σ_min·aspect-peak 인용 금지.**
6. **바이스태틱 σ는 β≲90°만 유효**(전방산란 σ≡0). 근거리·큰 L 은 걸림 — 해칭·수치금지. 상반성 깊은널 rms 5~9dB.
7. **ECA는 무한이상화가 아니다.** 정본 ridge_rel=0 은 측정상 사실상 이상적(잔류≈0). 진짜 낙관은 **실장 유한 소거깊이**(통상 40~90dB)·위상잡음·상호변조가 모델에 없다는 것 → `dpi_residual` 감도축으로 정량화.
8. **ADC는 1차근사.** 균일양자화+백색. AGC·SFDR·상호변조 미모델. '12-bit 가 벽'이라 단정 안 함(조건부).
9. **검출기 형상 둘·성격 다름.** S-G(게이트)=탐색없는 거리, S-W(전체창)=CPI당 오경보 수(분해능셀 기준). 검출정의=표적근방 문턱초과(전역 argmax 아님).
10. **N^¼ 이득은 이상적 상한**(완전코히런트·완벽조향·무상관). 개구는 밴드마다 다름.
11. **기준=full-waveform capture 정본** — 패시브인데 상시신호만은 아니라는 유보 승계(pilot-only 병기).
12. **CAF 2채널 승격 안 함** — 이상적 레퍼런스 정합필터 SNR(report12 동일 열린과제).
13. **문헌 실적치와 우열비교 안 함**(`prior_compare` 조건열 강제).
14. **호버·저속·앨리어싱 미검출은 거리무관.** 유휴 5G SSB(PRF50Hz)는 v=5m/s 전 헤딩 접힘 — 거리 이전에 도플러로 죽는다(하한 hover-blind + 상한 나이퀴스트 둘 다).
15. **자세=수평(roll=pitch=0)·yaw만.** 실전 진비행 pitch 10~30° 미모델 — **모른다**.
16. **el 격자 보간**: el≤−15° 는 β>90° 겹쳐 표본 적음. 헤드라인 el≈−2° 라 부호정정 효과는 헤드라인서 작다(d≲400m 한정). el>0(공중조명원) 범위 밖.
17. **원거리장**: s1000plus@5.21G 63.1m 최악. 그 아래 σ 인용 안 함.
18. **'5기종×9모드 한 규약 선행사례 확인 못함'까지만** — 부재증명 아님, '최초' 안 씀.
19. **단일 CPI·단일 표적.** 스캔누적·M-of-N·트래킹 없음(future work). 상호그림자 없음.
20. **대기감쇠·전파수평선은 숫자와 함께 무시**(FS-1엔 항 자체 없음, FS-3서만): 산소흡수 5km<0.1dB, 수평선≈39km≫20km.
21. **비단조·천장.** SNR(d)는 φ의 다수서 내부최대 — 이분법 무효, 최외곽교차로 해결. 천장보다 어두운 헤딩은 어떤 거리서도 미검출(거리문제 아니라 해 없음).
22. **비선형.** R∝σ^¼ 는 R_eq 축서만 정확. d 축 국소지수 1.03~4.00 — dB↔배율 환산 금지. k_mode(곱셈상수)는 이 기하 비선형 흡수 못함.
23. **동일채널 간섭** 한 항으로 거리 한자릿수 감소 가능(INR20dB 이웃셀) — 감도축으로만, 정본엔 미포함.
24. **파형↔반송파 결속.** 등전력만으론 파형효과 미분리 — 공통3.5GHz 반사실로 5항 분해. '어느 파형이 멀리 보나'는 밴드·σ(f)·반복률·fs선택의 합성이다.

> ### ▶ 다음 일 (future work): 추적
> 이 리포트는 **탐지**까지다. 위치·궤적을 잇는 **추적**은 감시 배열의 각도(AoA)로 3D 관측가능성을 확보해야 한다(report11). 본 실험이 쓴 다중 수신기 배열이 그 출발점이다.